In [1]:
import tensorflow as tf
from tensorflow import keras
import numpy as np

print("TF", tf.__version__)

TF 2.21.0


Fashion MNIST is built into Keras, so we load it directly.

In [2]:
(X_train_full, y_train_full), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()

X_train_full = X_train_full.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

X_valid, X_train = X_train_full[:5000], X_train_full[5000:]
y_valid, y_train = y_train_full[:5000], y_train_full[5000:]

print(X_train.shape, X_valid.shape, X_test.shape)

(55000, 28, 28) (5000, 28, 28) (10000, 28, 28)


Split the network into a lower block and an upper block so each can use its own optimizer.

In [3]:
lower_layers = keras.Sequential([
    keras.layers.Input(shape=[28, 28]),
    keras.layers.Flatten(),
    keras.layers.Dense(100, activation="relu"),
    keras.layers.Dense(100, activation="relu"),
])

upper_layers = keras.Sequential([
    keras.layers.Dense(10, activation="softmax"),
])

model = keras.Sequential([lower_layers, upper_layers])
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential (Sequential)         │ (None, 100)            │        88,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_1 (Sequential)       │ (None, 10)             │         1,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 89,610 (350.04 KB)

 Trainable params: 89,610 (350.04 KB)

 Non-trainable params: 0 (0.00 B)

Two optimizers: a slow SGD for the lower block and a faster Nadam for the upper block.

In [4]:
lower_optimizer = keras.optimizers.SGD(learning_rate=1e-4)
upper_optimizer = keras.optimizers.Nadam(learning_rate=1e-3)

Training config, metrics, and two helper functions for batching and printing progress.

In [5]:
n_epochs = 5
batch_size = 32
n_steps = len(X_train) // batch_size

loss_fn = keras.losses.sparse_categorical_crossentropy
mean_loss = keras.metrics.Mean(name="loss")
accuracy = keras.metrics.SparseCategoricalAccuracy(name="accuracy")


def random_batch(X, y, batch_size=32):
    idx = np.random.randint(len(X), size=batch_size)
    return X[idx], y[idx]


def print_status_bar(step, total, loss, metrics=None):
    metrics_str = " - ".join([f"{m.name}: {m.result():.4f}" for m in [loss] + (metrics or [])])
    end = "" if step < total else "\n"
    print(f"\r{step}/{total} - {metrics_str}", end=end)

Custom training loop. Prints the epoch, iteration, mean loss, and accuracy each step, then the validation loss and accuracy at the end of each epoch.

In [6]:
for epoch in range(1, n_epochs + 1):
    print(f"Epoch {epoch}/{n_epochs}")
    for step in range(1, n_steps + 1):
        X_batch, y_batch = random_batch(X_train, y_train, batch_size)

        with tf.GradientTape() as tape:
            y_pred = model(X_batch, training=True)
            main_loss = tf.reduce_mean(loss_fn(y_batch, y_pred))
            loss = tf.add_n([main_loss] + model.losses)

        # One set of gradients, applied by two optimizers
        gradients = tape.gradient(loss, model.trainable_variables)
        n_lower = len(lower_layers.trainable_variables)
        lower_optimizer.apply_gradients(zip(gradients[:n_lower], lower_layers.trainable_variables))
        upper_optimizer.apply_gradients(zip(gradients[n_lower:], upper_layers.trainable_variables))

        mean_loss(loss)
        accuracy(y_batch, y_pred)
        print_status_bar(step, n_steps, mean_loss, [accuracy])

    # Validation at the end of the epoch
    y_valid_pred = model(X_valid, training=False)
    val_loss = tf.reduce_mean(loss_fn(y_valid, y_valid_pred))
    val_acc = keras.metrics.SparseCategoricalAccuracy()(y_valid, y_valid_pred)
    print(f"Validation loss: {val_loss:.4f} - Validation accuracy: {val_acc:.4f}")

    mean_loss.reset_state()
    accuracy.reset_state()

Epoch 1/5
1718/1718 - loss: 1.1377 - accuracy: 0.6640
Validation loss: 0.7724 - Validation accuracy: 0.7490
Epoch 2/5
1718/1718 - loss: 0.7246 - accuracy: 0.7518
Validation loss: 0.6511 - Validation accuracy: 0.7758
Epoch 3/5
1718/1718 - loss: 0.6442 - accuracy: 0.7758
Validation loss: 0.5965 - Validation accuracy: 0.7960
Epoch 4/5
1718/1718 - loss: 0.6058 - accuracy: 0.7869
Validation loss: 0.5649 - Validation accuracy: 0.8080
Epoch 5/5
1718/1718 - loss: 0.5673 - accuracy: 0.7984
Validation loss: 0.5421 - Validation accuracy: 0.8164


Check accuracy on the test set.

In [7]:
y_test_pred = model(X_test, training=False)
test_acc = keras.metrics.SparseCategoricalAccuracy()(y_test, y_test_pred)
print(f"Test accuracy: {test_acc:.4f}")

Test accuracy: 0.7900
